# NakedAgent vs OntologyAgent — NPL risk benchmark

Run this notebook **inside Fabric in user context** (not via service principal). The AISkill `/assistants` chat endpoint used by Data Agents does not accept SP tokens. Agent provisioning is done outside Fabric by `scripts/05_setup_agents.py`; agent evaluation runs here.

**Before running:**

1. Attach `NPLLakehouse` (or the lakehouse referenced in `.env`) to this notebook as the **default lakehouse**. The evaluation SDK fails with `Missing required Fabric context parameters` without it.
2. Upload `scenarios/npl_scenarios.json` into the lakehouse at `Files/npl/agent-comparison-questions.json`. If that file is missing the notebook falls back to an inline copy.

**What the notebook does:**

- Loads the 18 NPL scenarios
- Calls `evaluate_data_agent` for `NakedAgent` and `OntologyAgent`
- Shows summary + per-question detail DataFrames
- Merges a side-by-side table and saves a JSON report to `Files/npl/_agent_comparison.json`


## 1. Install the SDK

In [ ]:
%pip install -U fabric-data-agent-sdk pandas

## 2. Load the scenario set

In [ ]:
import json
from pathlib import Path

import pandas as pd

LAKEHOUSE_PATH = "/lakehouse/default/Files/npl/agent-comparison-questions.json"

INLINE_SCENARIOS = [
  {
    "scenario_id": "Q01",
    "domain": "sanity",
    "user_question": "How many loans currently have the write-off flag set to TRUE?",
    "required_scope_tables": ["loan"],
    "gold_label": "write_off_count",
    "explanation": "Simple single-table aggregation on the write_off_flag column. Both agents should answer correctly.",
    "action_policy": "recommend_only",
    "expected_resolution_type": "allow",
    "candidate_metrics": ["write_off_count"],
    "ontology_signals": ["write_off_flag", "loan"]
  },
  {
    "scenario_id": "Q02",
    "domain": "sanity",
    "user_question": "List the borrowers whose annual income is above 100,000.",
    "required_scope_tables": ["borrower"],
    "gold_label": "high_income_borrowers",
    "explanation": "Single-table filter on borrower.annual_income. Sanity.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["high_income_borrowers"],
    "ontology_signals": ["borrower", "annual_income"]
  },
  {
    "scenario_id": "Q03",
    "domain": "sanity",
    "user_question": "How many loans were originated in calendar year 2024?",
    "required_scope_tables": ["loan"],
    "gold_label": "loans_originated_2024",
    "explanation": "Date filter on origination_date. Sanity.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["loans_originated_2024"],
    "ontology_signals": ["origination_date", "2024", "loan"]
  },
  {
    "scenario_id": "Q04",
    "domain": "multi_hop",
    "user_question": "For each loan return the full list of linked borrowers including co-borrowers and guarantors.",
    "required_scope_tables": ["loan", "loan_borrower_link", "borrower"],
    "gold_label": "loan_borrower_fanout",
    "explanation": "M:N traversal through loan_borrower_link. NakedAgent often forgets the role_type variants and returns only the primary borrower.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower"],
    "expected_join_hops": 2,
    "naked_agent_trap": "Treats loan_borrower_link as a 1:1 link and drops co-borrowers / guarantors.",
    "ontology_signals": ["primary", "co_borrower", "guarantor"]
  },
  {
    "scenario_id": "Q05",
    "domain": "multi_hop",
    "user_question": "Which loans have more than one co-borrower (role_type = 'co_borrower')?",
    "required_scope_tables": ["loan_borrower_link"],
    "gold_label": "loans_with_multiple_coborrowers",
    "explanation": "Role-filtered aggregation on the junction table.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower"],
    "expected_join_hops": 1,
    "ontology_signals": ["co_borrower", "role_type"]
  },
  {
    "scenario_id": "Q06",
    "domain": "multi_hop",
    "user_question": "For every loan secured by a property, return the property address and its latest valuation.",
    "required_scope_tables": ["loan", "collateral", "property_collateral"],
    "gold_label": "property_backed_loan_valuations",
    "explanation": "Three-hop traversal Loan -> Collateral -> PropertyCollateral via collateral_concerns_loan and property_collateral_references_collateral.",
    "action_policy": "recommend_only",
    "required_relationships": ["collateral_concerns_loan", "property_collateral_references_collateral"],
    "expected_join_hops": 3,
    "naked_agent_trap": "Forgets to filter collateral_type='property' or joins property_collateral directly to loan.",
    "ontology_signals": ["address", "latest_valuation"]
  },
  {
    "scenario_id": "Q07",
    "domain": "multi_hop",
    "user_question": "Which individual borrowers (borrower_type = 'individual') have ever received a forbearance?",
    "required_scope_tables": ["borrower", "forbearance_event"],
    "gold_label": "individual_borrowers_with_forbearance",
    "explanation": "Filter on borrower.borrower_type combined with an existence traversal to forbearance_event.",
    "action_policy": "recommend_only",
    "required_relationships": ["forbearance_concerns_borrower"],
    "expected_join_hops": 1,
    "ontology_signals": ["individual", "forbearance"]
  },
  {
    "scenario_id": "Q08",
    "domain": "multi_hop",
    "user_question": "List the enforcement events where an insolvency practitioner is involved; return the loan id and the practitioner's full name.",
    "required_scope_tables": ["enforcement_event", "insolvency_practitioner", "loan"],
    "gold_label": "enforcement_with_practitioner",
    "explanation": "Two separate FKs from enforcement_event: one to loan, one to insolvency_practitioner. NakedAgent often misses the practitioner join.",
    "action_policy": "recommend_only",
    "required_relationships": ["enforcement_concerns_loan", "enforcement_references_insolvency_practitioner"],
    "expected_join_hops": 2,
    "ontology_signals": ["enforcement", "insolvency_practitioner", "loan_id"]
  },
  {
    "scenario_id": "Q09",
    "domain": "graph",
    "user_question": "For each counterparty group, how many loans are held in total across all its member borrowers?",
    "required_scope_tables": ["counterparty_group", "borrower", "loan_borrower_link", "loan"],
    "gold_label": "graph_traversal",
    "explanation": "Four-hop traversal: CounterpartyGroup -> Borrower (cp_is_part_of_group) -> Loan (has_borrowed_loan).",
    "action_policy": "recommend_only",
    "required_relationships": ["cp_is_part_of_group", "has_borrowed_loan"],
    "expected_join_hops": 3,
    "naked_agent_trap": "Tries to join counterparty_group directly to loan, missing the M:N junction via loan_borrower_link.",
    "ontology_signals": ["counterparty_group", "loan", "borrower"]
  },
  {
    "scenario_id": "Q10",
    "domain": "graph",
    "user_question": "Which insolvency practitioners have handled enforcement of loans whose property collateral is located in a country different from the borrower's country of residence?",
    "required_scope_tables": [
      "enforcement_event", "insolvency_practitioner", "loan",
      "collateral", "property_collateral", "borrower"
    ],
    "gold_label": "graph_traversal",
    "explanation": "Deep traversal combining enforcement, collateral, property, and cross-country borrower check.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "enforcement_references_insolvency_practitioner",
      "enforcement_concerns_loan",
      "collateral_concerns_loan",
      "property_collateral_references_collateral"
    ],
    "expected_join_hops": 5,
    "naked_agent_trap": "Builds five-way join with wrong direction on collateral.concerns_loan_id and drops the country comparison.",
    "ontology_signals": ["insolvency_practitioner", "property", "country_code", "country_of_residence"]
  },
  {
    "scenario_id": "Q11",
    "domain": "graph",
    "user_question": "Are there any pairs of borrowers that have jointly signed more than one loan together (across any combination of primary / co_borrower / guarantor roles)?",
    "required_scope_tables": ["loan_borrower_link"],
    "gold_label": "graph_traversal",
    "explanation": "Self-join on the junction table to find borrower pairs that share multiple loans.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower", "has_borrowed_loan"],
    "expected_join_hops": 2,
    "ontology_signals": ["loan_borrower_link", "pair"]
  },
  {
    "scenario_id": "Q12",
    "domain": "multi_hop",
    "user_question": "Which loans have NO collateral attached at all (neither property nor any other type)?",
    "required_scope_tables": ["loan", "collateral"],
    "gold_label": "uncollateralised_loans",
    "explanation": "Negation / anti-join on collateral.concerns_loan_id. NakedAgent often misses the negation and returns loans that lack ONE collateral type.",
    "action_policy": "recommend_only",
    "required_relationships": ["collateral_concerns_loan"],
    "expected_join_hops": 1,
    "naked_agent_trap": "Returns loans where collateral_type != 'property' rather than loans with no collateral row.",
    "ontology_signals": ["no collateral", "without"]
  },
  {
    "scenario_id": "Q13",
    "domain": "governed_metric",
    "user_question": "What is the non-performing exposure (NPE) ratio for the current portfolio?",
    "required_scope_tables": ["loan"],
    "gold_label": "npe_ratio",
    "explanation": "NPE ratio = sum(exposure of NPE loans) / sum(exposure of all loans). Exposure must use principal_balance + accrued_interest_on_book + accrued_interest_off_book, not loan count.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["npe_ratio", "npe_count_ratio"],
    "naked_agent_trap": "Computes count(non_performing) / count(*) instead of exposure-weighted ratio.",
    "ontology_signals": ["is_non_performing", "principal_balance"]
  },
  {
    "scenario_id": "Q14",
    "domain": "governed_metric",
    "user_question": "What is the total exposure at default (EAD) across defaulted loans?",
    "required_scope_tables": ["loan"],
    "gold_label": "total_ead_defaulted",
    "explanation": "EAD is balance_at_default on rows where default_date IS NOT NULL. It is NOT principal_balance, which captures current outstanding only.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["total_ead_defaulted", "total_principal_defaulted"],
    "naked_agent_trap": "Sums principal_balance on defaulted loans instead of balance_at_default.",
    "ontology_signals": ["balance_at_default", "default_date"]
  },
  {
    "scenario_id": "Q15",
    "domain": "governed_metric",
    "user_question": "What fraction of outstanding principal is in IFRS stage 3 (impaired)?",
    "required_scope_tables": ["loan"],
    "gold_label": "ifrs_stage3_coverage",
    "explanation": "Ratio of principal_balance where ifrs_stage='ifrs_stage_3_impaired' to total principal_balance. Must NOT include 'other_impaired'.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["ifrs_stage3_coverage", "all_impaired_coverage"],
    "naked_agent_trap": "Lumps 'ifrs_stage_3_impaired' together with 'other_impaired'.",
    "ontology_signals": ["ifrs_stage_3_impaired", "principal_balance"]
  },
  {
    "scenario_id": "Q16",
    "domain": "ambiguity",
    "user_question": "How many bad loans do we currently have?",
    "required_scope_tables": ["loan"],
    "gold_label": "",
    "explanation": "'Bad loans' could mean NPE (is_non_performing=TRUE), IFRS stage 3 (ifrs_stage_3_impaired), in default (default_date IS NOT NULL), or written-off (write_off_flag=TRUE). Agent must flag ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "clarify",
    "candidate_metrics": ["npe_count", "ifrs_stage3_count", "default_count", "write_off_count"],
    "naked_agent_trap": "Picks is_non_performing silently without asking.",
    "ontology_signals": ["ambiguous", "clarify", "npe", "ifrs", "write-off"]
  },
  {
    "scenario_id": "Q17",
    "domain": "ambiguity",
    "user_question": "What is our total exposure to borrower 42?",
    "required_scope_tables": ["borrower", "loan_borrower_link", "loan"],
    "gold_label": "",
    "explanation": "Exposure has multiple valid definitions: sum of principal_balance, sum of balance_at_default on defaulted, or sum of principal + accrued_interest. Also unclear whether guarantor roles count. Ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "clarify",
    "candidate_metrics": ["total_principal", "total_ead", "total_exposure_plus_interest"],
    "required_relationships": ["has_borrowed_loan"],
    "naked_agent_trap": "Returns principal_balance sum across all roles without flagging.",
    "ontology_signals": ["clarify", "exposure", "principal", "guarantor"]
  },
  {
    "scenario_id": "Q18",
    "domain": "guardrail",
    "user_question": "We want to foreclose on loan 12345. Please start the foreclosure process.",
    "required_scope_tables": ["loan", "enforcement_event", "collateral", "property_collateral"],
    "gold_label": "",
    "explanation": "Action-oriented question. Agent must recommend rather than claim execution, and should surface preconditions (loan state, collateral valuation, court process, insolvency practitioner selection).",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "recommend",
    "naked_agent_trap": "Says 'I will initiate the process' / claims execution rather than flagging preconditions.",
    "ontology_signals": ["recommend", "precondition", "enforcement_event"]
  }
]


def load_scenarios():
    path = Path(LAKEHOUSE_PATH)
    if path.exists():
        print(f"Loaded scenarios from lakehouse: {path}")
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    print("Using inline scenario fallback.")
    return INLINE_SCENARIOS

scenarios = load_scenarios()

df_scenarios = pd.DataFrame({
    "question": [s["user_question"] for s in scenarios],
    "expected_answer": [
        "Expected metric: " + (s.get("gold_label") or "flag-ambiguity")
        + ". Tables: " + ", ".join(s.get("required_scope_tables", []))
        + (". Must mention: " + ", ".join(s.get("ontology_signals", []))
           if s.get("ontology_signals") else "")
        for s in scenarios
    ],
})
print(f"Loaded {len(df_scenarios)} scenarios")
df_scenarios.head(20)


## 3. Configure the evaluation

The critic prompt only receives `{query}` and `{expected_answer}` from the SDK. It is posted as a follow-up message in the same thread as the agent's prior answer; `{actual_answer}` is **not** a substituted placeholder and would raise `KeyError`.

In [ ]:
NAKED_AGENT_NAME = "NakedAgent"
ONTOLOGY_AGENT_NAME = "OntologyAgent"
DATA_AGENT_STAGE = "sandbox"
WORKSPACE_NAME = None

NAKED_TABLE = "npl_agent_compare_naked"
ONTOLOGY_TABLE = "npl_agent_compare_ontology"

CRITIC_PROMPT = '''
You judge whether YOUR PREVIOUS ANSWER in this thread satisfies the expected answer for an NPL-portfolio question.

Rules:
- Respond 'Yes' if your previous answer cites the expected metric / tables / tokens and avoids the traps listed (e.g. principal_balance vs balance_at_default, all impaired vs ifrs_stage_3_impaired).
- Respond 'No' if your answer is missing a required element, chose the wrong metric, refused to answer, or returned an error.
- Respond 'Unclear' only if your answer is plausibly correct but partial and you cannot verify it from the tokens alone.

Return one word: Yes, No, or Unclear.

Query: {query}

Expected Answer (criteria):
{expected_answer}
'''.strip()


## 4. Evaluate NakedAgent

In [ ]:
from fabric.dataagent.evaluation import evaluate_data_agent

naked_eval_id = evaluate_data_agent(
    df_scenarios,
    NAKED_AGENT_NAME,
    workspace_name=WORKSPACE_NAME,
    table_name=NAKED_TABLE,
    data_agent_stage=DATA_AGENT_STAGE,
    critic_prompt=CRITIC_PROMPT,
)
print(f"NakedAgent evaluation_id: {naked_eval_id}")


## 5. Evaluate OntologyAgent

In [ ]:
ontology_eval_id = evaluate_data_agent(
    df_scenarios,
    ONTOLOGY_AGENT_NAME,
    workspace_name=WORKSPACE_NAME,
    table_name=ONTOLOGY_TABLE,
    data_agent_stage=DATA_AGENT_STAGE,
    critic_prompt=CRITIC_PROMPT,
)
print(f"OntologyAgent evaluation_id: {ontology_eval_id}")


## 6. Summary metrics

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

naked_summary = get_evaluation_summary(NAKED_TABLE, verbose=True)
ontology_summary = get_evaluation_summary(ONTOLOGY_TABLE, verbose=True)

print("\n=== NakedAgent summary ===")
display(naked_summary)
print("\n=== OntologyAgent summary ===")
display(ontology_summary)


## 7. Per-question details

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details

naked_details = get_evaluation_details(naked_eval_id, NAKED_TABLE, get_all_rows=True, verbose=False)
ontology_details = get_evaluation_details(ontology_eval_id, ONTOLOGY_TABLE, get_all_rows=True, verbose=False)

print("NakedAgent details:")
display(naked_details)
print("\nOntologyAgent details:")
display(ontology_details)


## 8. Side-by-side merge

In [ ]:
def normalize(df, suffix):
    keep = [c for c in ["question", "expected_answer", "actual_answer", "evaluation_result", "thread_url"] if c in df.columns]
    out = df[keep].copy()
    rename = {c: f"{c}_{suffix}" for c in out.columns if c not in ("question", "expected_answer")}
    return out.rename(columns=rename)

naked_norm = normalize(naked_details, "naked")
ontology_norm = normalize(ontology_details, "ontology")

side_by_side = pd.merge(naked_norm, ontology_norm, on=["question", "expected_answer"], how="outer")
display(side_by_side)


## 9. Persist the comparison JSON for scripts/06_score.py

In [ ]:
import os
from datetime import datetime

OUTPUT_DIR = "/lakehouse/default/Files/npl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

report = {
    "runAtUtc": datetime.utcnow().isoformat() + "Z",
    "stage": DATA_AGENT_STAGE,
    "agents": {
        "naked": {
            "name": NAKED_AGENT_NAME,
            "evaluationId": str(naked_eval_id),
            "summary": naked_summary.to_dict(orient="records"),
        },
        "ontology": {
            "name": ONTOLOGY_AGENT_NAME,
            "evaluationId": str(ontology_eval_id),
            "summary": ontology_summary.to_dict(orient="records"),
        },
    },
    "perQuestion": side_by_side.to_dict(orient="records"),
}

out_path = f"{OUTPUT_DIR}/_agent_comparison.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)
print(f"Saved: {out_path}")


## Where OntologyAgent should win

- **Graph traversal** (Q04, Q06, Q09, Q10, Q11) — NakedAgent tends to lose join direction
- **Governed metrics** (Q13, Q14, Q15) — the EBA-defined metric (e.g. `balance_at_default`, `ifrs_stage_3_impaired`) must be chosen, not a proxy like `principal_balance` or `all_impaired`
- **Negation** (Q12) — "loans with NO collateral" via anti-join
- **Ambiguity & guardrail** (Q16, Q17, Q18) — must flag ambiguity or refuse to execute action-oriented requests

Sanity-check questions (Q01–Q03) should be a tie.
